In [8]:
!pip install transformers

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

[transformers] BertForMaskedLM LOAD REPORT from: dbmdz/bert-base-german-cased
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForMaskedLM

# 1. Modell und Tokenizer laden
MOD_NAME = "dbmdz/bert-base-german-cased"
tok = AutoTokenizer.from_pretrained(MOD_NAME)
model = AutoModelForMaskedLM.from_pretrained(MOD_NAME, output_attentions=True, output_hidden_states=True)
model.eval()

# ==============================================================================
# 🎯 SZENARIO-AUSWAHL: Wähle "wanderer" oder "katze"
# ==============================================================================
SZENARIO = "wanderer"  # Alternativ: "katze"

KONFIGURATION = {
    "wanderer": {
        "satz": "Der Wanderer setzt sich auf die [MASK] .",
        "schritte": [
            ("Schritt 0: Kein Kontext (Isoliertes [MASK])", []),
            ("Schritt 1: + Grammatik ('die')", ["die"]),
            ("Schritt 2: + Handlung ('setzt')", ["die", "setzt"]),
            ("Schritt 3: + Subjekt ('Wanderer')", ["die", "setzt", "Wanderer"]),
            ("Schritt 4: + Richtung ('auf')", ["die", "setzt", "Wanderer", "auf"]),
            ("Schritt 5: Voller Satz (100% Kontext)", ["Der", "Wanderer", "setzt", "sich", "auf", "die", "."]),
        ]
    },
    "katze": {
        "satz": "Die Katze setzt sich auf die [MASK] .",
        "schritte": [
            ("Schritt 0: Kein Kontext (Isoliertes [MASK])", []),
            ("Schritt 1: + Grammatik ('die')", ["die"]),
            ("Schritt 2: + Handlung ('setzt')", ["die", "setzt"]),
            ("Schritt 3: + Subjekt ('Katze')", ["die", "setzt", "Katze"]),
            ("Schritt 4: + Richtung ('auf')", ["die", "setzt", "Katze", "auf"]),
            ("Schritt 5: Voller Satz (100% Kontext)", ["Die", "Katze", "setzt", "sich", "auf", "die", "."]),
        ]
    }
}

# 2. Satz tokenisieren
aktuell = KONFIGURATION[SZENARIO]
satz = aktuell["satz"]
inputs = tok(satz, return_tensors="pt")
input_ids = inputs["input_ids"]
tokens = tok.convert_ids_to_tokens(input_ids[0])

mask_idx = (input_ids[0] == tok.mask_token_id).nonzero(as_tuple=True)[0].item()
cls_idx = 0
sep_idx = len(tokens) - 1
schritte_plan = aktuell["schritte"]
schritt = 0

print(f"✅ Szenario '{SZENARIO.upper()}' geladen!")
print(f"📖 Satz: \"{satz}\"")
print(f"🔢 Tokens: {tokens}")
print("👉 Führe jetzt Zelle 2 mehrfach mit 'Shift + Enter' aus.")

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

[transformers] BertForMaskedLM LOAD REPORT from: dbmdz/bert-base-german-cased
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.weight | UNEXPECTED |  | 
bert.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Setup geladen! Satz: "Der Wanderer setzt sich auf die [MASK] ."
👉 Führe jetzt Zelle 2 mehrfach mit 'Shift + Enter' aus.


In [ ]:
# ==============================================================================
# INTERAKTIVER SCHRITT: Shift + Enter drücken zum Weiterschalten
# ==============================================================================

with torch.no_grad():
    titel, aktive_woerter = schritte_plan[schritt]

    # Attention-Maske aufbauen: CLS, SEP und MASK immer aktiv
    att_mask = torch.zeros_like(input_ids)
    att_mask[0, cls_idx] = 1
    att_mask[0, sep_idx] = 1
    att_mask[0, mask_idx] = 1
    
    for idx, t in enumerate(tokens):
        if t in aktive_woerter:
            att_mask[0, idx] = 1

    # Vorwärtspass
    outputs = model(input_ids=input_ids, attention_mask=att_mask)
    
    # 1. Q, K, V aus Schicht 0 extrahieren
    embeds = model.bert.embeddings(input_ids)
    att_layer = model.bert.encoder.layer[0].attention.self
    Q = att_layer.query(embeds)[0]
    K = att_layer.key(embeds)[0]
    V = att_layer.value(embeds)[0]
    
    q_mask = Q[mask_idx]
    d_k = q_mask.shape[-1]
    q_preview = ", ".join([f"{val:.2f}" for val in q_mask[:10].tolist()])

    # 2. Scores & Softmax (A) über sichtbare Tokens
    aktive_indices = (att_mask[0] == 1).nonzero(as_tuple=True)[0].tolist()
    raw_scores = {idx: (q_mask @ K[idx]).item() / (d_k ** 0.5) for idx in aktive_indices}
    exp_scores = {idx: torch.exp(torch.tensor(s)) for idx, s in raw_scores.items()}
    sum_exp = sum(exp_scores.values())
    A = {idx: (exp / sum_exp).item() for idx, exp in exp_scores.items()}

    # 3. Residual-Verschiebung (Δ) in Schicht 0
    h_embed = outputs.hidden_states[0][0, mask_idx]
    h_layer0 = outputs.hidden_states[1][0, mask_idx]
    delta_norm = torch.norm(h_layer0 - h_embed).item()

    # 4. Vorhersagen für [MASK]
    logits = outputs.logits[0, mask_idx]
    probs = torch.softmax(logits, dim=-1)
    top_p, top_i = torch.topk(probs, 3)
    pred_str = ", ".join([f"'{tok.decode([idx]).strip()}' ({p.item()*100:.1f}%)" for p, idx in zip(top_p, top_i)])

    # Lesbare Satzanzeige für Studierende
    sichtbar = " ".join([
        tokens[i] if (t in aktive_woerter or t == "[MASK]") else "___"
        for i, t in enumerate(tokens)
        if t not in ["[CLS]", "[SEP]"]
    ])

    print("=" * 76)
    print(f"📌 {titel}")
    print(f"📖 Sichtbarer Kontext          : \"{sichtbar}\"")
    print("-" * 76)
    print(f"🔍 EMPFÄNGER (Q_[MASK])        : 768-D Vektor (Form: x_[MASK] · W^Q)")
    print(f"   Vektor-Ausschnitt Q (0..9)  : [{q_preview}, ...]")
    print("-" * 76)
    print("🏷️  SENDER-ABGLEICH (Q · K -> A & V):")
    for idx in aktive_indices:
        t_name = tokens[idx]
        if t_name in ["[CLS]", "[SEP]"]:
            continue
        v_norm = torch.norm(V[idx]).item()
        print(f"   • {t_name:<10} | Score (Q·K): {raw_scores[idx]:5.2f} | Gewicht A: {A[idx]*100:5.1f} % | V-Norm: {v_norm:.1f}")
    
    print("-" * 76)
    value_terms = [f"{A[i]*100:.1f}%·V_{tokens[i]}" for i in aktive_indices if tokens[i] not in ["[MASK]", "[CLS]", "[SEP]"]]
    print(f"📦 VALUE-MISCHUNG (∑ A·V)     : " + (" + ".join(value_terms) if value_terms else "(keine externen Values)"))
    print(f"⚡ RESIDUAL-UPDATE (+ Δ)      : ‖Δ‖ = {delta_norm:.2f}  (H_neu = H_alt + Δ)")
    print(f"🎯 TOP-3 VORHERSAGEN          : {pred_str}")
    print("=" * 76)

    if schritt < len(schritte_plan) - 1:
        schritt += 1
        print(f"👉 Nächster Schritt: {schritte_plan[schritt][0]} (Shift + Enter drücken)")
    else:
        print("✅ Alle Schritte abgeschlossen! Nächstes Shift + Enter setzt zurück auf Schritt 0.")
        schritt = 0

📌 Schritt 0: Kein Kontext (Isoliertes [MASK])
📖 Sichtbarer Kontext          : "___ ___ ___ ___ ___ ___ [MASK] ___"
----------------------------------------------------------------------------
🔍 EMPFÄNGER (Q_[MASK])        : 768-D Vektor (Form: x_[MASK] · W^Q)
   Vektor-Ausschnitt Q (Dim 0..9): [-0.67, 0.04, 0.10, 1.02, 0.32, -0.08, -0.23, 0.11, -0.08, 0.86, ...]
----------------------------------------------------------------------------
🏷️  SENDER-ABGLEICH (Q · K -> A & V):
   • [MASK]     | Score (Q·K): 16.47 | Gewicht A: 100.0 % | V-Norm: 10.8
----------------------------------------------------------------------------
📦 VALUE-MISCHUNG (∑ A·V)     : 
⚡ RESIDUAL-UPDATE (+ Δ)      : ‖Δ‖ = 9.80  (H_neu = H_alt + Δ)
🎯 TOP-3 VORHERSAGEN          : '##e' (6.7%), '##r' (4.8%), '-' (3.3%)
👉 Nächster Schritt: Schritt 1: + Grammatik ('die') (Shift + Enter drücken)


### 📝 Übungsaufgabe: Der Informationsfluss in der Self-Attention

Führe Zelle 2 Schritt für Schritt mit `Shift + Enter` aus und bearbeite die folgenden Aufgaben.

---

#### Teil 1: Protokolliere die Vektor-Verschiebung
Fülle die Tabelle anhand deiner Ausgaben aus:

| Schritt | Neu aufgedeckter Key | Höchstes Gewicht $A$ (Sender) | Top-1 Vorhersage (Wort & %) | Stärke $\|\|\Delta\|\|$ |
| :--- | :--- | :--- | :--- | :--- |
| **0** | *(keiner)* | `[MASK]` ($100\,\%$) | | |
| **1** | `'die'` | | | |
| **2** | `'setzt'` | | | |
| **3** | `'Wanderer'` | | | |
| **4** | `'auf'` | | | |
| **5** | *Voller Satz* | | | |

---

#### Teil 2: Analysefragen (Mechanismus verstehen)


0. **Warum sind die ersten 10 Werte von $Q_{\text{[MASK]}}$ in allen Schritten exakt identisch `[-0.67, 0.04, 0.10, ...]`?**  
  
1. **Die Filterwirkung von $V_{\text{die}}$ (Schritt 1):**
   Welche Wortarten bzw. welches grammatikalische Geschlecht (Genus) dominieren die Vorhersagen nach Schritt 1? Warum scheiden Wörter wie *„Stuhl“* oder *„Tisch“* hier mathematisch aus?

2. **Die semantische Verdrängung (Schritt 2 vs. Schritt 3):**
   * Welche Vorhersagen tauchen in Schritt 2 auf, sobald $V_{\text{setzt}}$ einfließt?

3. **Routing vs. Payload:**
   Erkläre den Unterschied anhand von Schritt 3:
   * Welche Rolle spielen $Q_{\text{[MASK]}}$ und $K_{\text{Wanderer}}$?
   * Welche Rolle spielt $V_{\text{Wanderer}}$?

4. **Der Residual-Vektor $\Delta$:**
   In der Ausgabe siehst du $\|\|\Delta\|\|$. Was passiert mathematisch mit dem Zustand des Tokens `[MASK]`, wenn $\Delta$ im Residual Stream aufaddiert wird ($H_{\text{neu}} = H_{\text{alt}} + \Delta$)?

---

#### Teil 3: Transfer-Experiment (Code-Challenge)
Ändere in Zelle 1 das Subjekt von `„Der Wanderer“` zu `„Die Katze“` und lasse die Schritte erneut durchlaufen:
* Welches Wort landet nun in Schritt 5 auf Platz 1?
* Anhand welcher Value-Vektoren ($V$) lässt sich dieser Wechsel erklären?